# Does the soil decide what farmers plant?

**The question.** Farmers do not plant anything anywhere. Soil
properties - texture, water storage, organic matter - decide which crops
are economically viable on a given field. Bare-soil *reflectance* is a
window into exactly those properties: its colour tracks organic matter,
iron oxides and texture.

So we ask the question in the right direction: **does the soil's spectral
signature (its colour) predict the crop a farmer grows there?**

**The data.** Two very different cloud-native products from Thünen:

* a **categorical time series** - annual crop-type maps (2013-2023)
  that tell us *what was grown*,
* a **single 29 GB raster** - one nationwide bare-soil reflectance
  composite (Sentinel-2, 2015-2024) that tells us *what the soil looks
  like*, as a single Cloud-Optimized GeoTIFF.

**The plan.** Find both with STAC, collapse 11 crop maps into the
*majority crop per pixel*, stream the soil composite onto the same grid,
and ask whether the soil spectrum separates the crops.

## The plan

| Step | What we do | Cloud-native idea |
| --- | --- | --- |
| 1. Find | search crop maps + soil in the STAC API | metadata first, no download |
| 2. Majority crop | mode over 11 years with `apply_ufunc` | run any function, block by block |
| 3. Stream soil | read a 29 GB COG for one window | COG overviews + geobox alignment |
| 4. Compare | do soil spectra separate the crops? | all on a laptop |

As before, every step is written out explicitly.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import pystac_client
import xarray as xr

import eo_utils as eo

eo.apply_cloud_defaults()
print("Cloud-friendly GDAL/Dask defaults applied.")

## Step 1 - Find the crop maps

We reuse the STAC search we met in notebook 01: open the Thünen
catalogue, filter by collection, bounding box and time, then load the
matching items as a **lazy** stack at 100 m (a deliberate speed/precision
trade-off for a live demo).

In [ ]:
thuenen = pystac_client.Client.open("https://eodata.thuenen.de/stac/api/v1/")

soil_bbox = [9.5, 51.9, 10.5, 52.6]      # Braunschweig loess belt
soil_years = "2013-01-01/2023-12-31"

crop_items = list(thuenen.search(
    collections=["hist-crop-type-map"], bbox=soil_bbox, datetime=soil_years
).items())
print("Crop-map years:", len(crop_items))

crop_stack = odc.stac.load(
    crop_items,
    bands=["crop_type"],
    bbox=soil_bbox,
    bbox_crs="EPSG:4326",
    crs="EPSG:3035",
    resolution=100,
    chunks={"time": -1, "x": 256, "y": 256},
    resampling="nearest",
    fail_on_error=False,
)
crop_stack

## Step 2 - Majority crop per pixel

We collapse 11 annual maps into one *most frequent crop* layer. Two
subtleties:

* `0` is the product's nodata ("not agricultural land"). We turn it into
  `NaN` so it can never win the vote.
* `xarray.apply_ufunc` with `dask="parallelized"` applies SciPy's `mode`
  **block by block**, with `time` as the reduction dimension. This is how
  you plug an arbitrary NumPy/SciPy function into a lazy array.

In [ ]:
from scipy.stats import mode

def _mode_ignore_nan(arr, axis=-1):
    return mode(arr, axis=axis, keepdims=False, nan_policy="omit").mode

crop_vals = crop_stack["crop_type"].where(crop_stack["crop_type"] != 0).astype("float32")

majority_lazy = xr.apply_ufunc(
    _mode_ignore_nan,
    crop_vals,
    input_core_dims=[["time"]],     # reduce over time
    output_core_dims=[[]],
    kwargs={"axis": -1},
    dask="parallelized",
    output_dtypes=["float32"],
).rename("majority_crop")

eo.dask_report(majority_lazy, "Majority crop per pixel (lazy)")

In [ ]:
with eo.compute_with_progress():
    majority = majority_lazy.compute()
majority

In [ ]:
crop_legend = eo.legend_from_item(crop_items[0])   # legend read from STAC

counts = majority.to_series().value_counts()
summary = pd.DataFrame(
    {"class": [crop_legend.loc[int(c), "title"] for c in counts.index],
     "pixels": counts.values},
    index=counts.index.astype(int),
)
summary

In [ ]:
eo.plot_classified_map(
    majority, crop_legend,
    title="Historical majority crop type (2013-2023)",
    savepath="majority_crop_map.png",
)

## Step 3 - Stream the bare-soil composite onto the same grid

The composite is a single 29 GB COG. We pass the **crop geobox** so
`odc-stac` reads a matching window and resamples it. We derive the band
names and nodata value from the item's `raster:bands` metadata - another
case where STAC tells us how to interpret the file.

COG **overviews** make this cheap: for 100 m output, GDAL reads a
decimated pyramid level instead of the full 10 m data.

In [ ]:
soil_items = list(thuenen.search(collections=["soil-reflectance-composite"],
                                 bbox=soil_bbox).items())
soil_item = soil_items[0]

band_meta = soil_item.properties["raster:bands"]
band_names = [b["name"] for b in band_meta]
nodata = band_meta[0]["nodata"]
print("Bands :", band_names)
print("Nodata:", nodata, "| advertised scale:", band_meta[0]["scale"])

soil_ds = odc.stac.load(
    soil_items,
    geobox=crop_stack.odc.geobox,     # <- align to the crop grid
    chunks={"x": 256, "y": 256},
    fail_on_error=False,
).squeeze(drop=True)

soil_ds = soil_ds.rename({f"data.{i + 1}": name for i, name in enumerate(band_names)})
# The metadata says scale=1.0, but the stored values are reflectance x 10000.
soil_ds = soil_ds.where(soil_ds != nodata) * 1e-4
soil_ds

In [ ]:
eo.dask_report(soil_ds, "Bare-soil composite (lazy)")
with eo.compute_with_progress():
    soil_ds = soil_ds.compute()
soil_ds[["red", "green", "blue"]]

## Step 4 - Does the soil differ between crops?

For each crop we take pixels where it is the historical majority, drop
nodata, trim the extreme 1% of each band (hot pixels and artefacts) and
stack everything into one tidy table. One row = one bare-soil pixel, so
we can compare the soil spectrum of fields that grow different crops.

In [ ]:
crop_codes = {"Maize": 130, "Rapeseed": 1501, "Potato": 1401, "Sugar Beet": 1402}
bands = band_names

frames = []
for name, code in crop_codes.items():
    masked = soil_ds[bands].where(majority == code)
    df = masked.to_dataframe().dropna()
    if df.empty:
        continue
    for b in bands:                       # trim extreme 1% per band
        lo, hi = df[b].quantile([0.01, 0.99])
        df = df[df[b].between(lo, hi)]
    df = df.reset_index(drop=True)
    df["Crop"] = name
    frames.append(df)

soil_df = pd.concat(frames, ignore_index=True)
print(f"{len(soil_df):,} bare-soil pixels across {soil_df['Crop'].nunique()} crops")
soil_df.groupby("Crop")[["red", "green", "blue"]].mean().round(4)

In [ ]:
eo.plot_mean_spectra(soil_df, bands, savepath="bare_soil_spectra_by_crop.png")

### The colour of the soil

We reduce each pixel to two intuitive colour coordinates:

* **x** = red / green (how "red" versus "green" the soil is),
* **y** = blue / (red + green + blue).

Every dot is coloured with the pixel's **true RGB** - we are literally
plotting the colour of the soil. The ellipses show one standard deviation
of each crop's distribution; the X marks the mean.

In [ ]:
soil_df["x_ax"] = soil_df["red"] / soil_df["green"]
soil_df["y_ax"] = soil_df["blue"] / (soil_df["red"] + soil_df["green"] + soil_df["blue"])

eo.plot_chromaticity(soil_df, savepath="soil_rgb_distribution_zoomed.png")

## What did we learn?

* The four crop distributions **overlap heavily**. That is expected:
  many crops grow on a wide range of soils, and the crop maps carry
  their own uncertainty.
* The separation we do see is consistent with **soil steering crop
  choice** - for example sugar beet on darker, moisture-retentive loess
  versus potato on lighter sandy soils. But this is observational, not
  causal proof: soil type and crop choice are both shaped by the same
  landscape, climate and management history.

### Why this only worked cloud-natively

* A **29 GB** national COG was queried for one ~70 km window in a couple
  of seconds thanks to overviews and range reads - we never downloaded it.
* The crop maps and soil were **aligned through geoboxes** on read, with
  no intermediate files.
* The band names, nodata value and legend all came from **STAC metadata**,
  so the analysis describes itself.

**Try it yourself:** swap `crop_codes` for `1603` (horticultural crops) or
`200` (grassland), add `swir1`/`swir2` to the chromaticity, or rerun at
30 m to sharpen the picture.